# PID-RL PoC — GRPO training in Colab

Trains **Qwen 2.5 1.5B Instruct** via GRPO to act as an autonomous governance agent for the GRIT stablecoin PID controller.

- Code: https://github.com/Grinta-Protocol/qwen-rl-shanghai
- Target: T4 GPU (Colab free tier)
- ETA: ~3–4h for 500 steps

**Pipeline:** `scenarios.py` → `prompt.py` → Qwen 2.5 1.5B → JSON → `parse_output` → `reward.py` → GRPO update.

## 1 · Environment check

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"bf16 supported: {torch.cuda.is_bf16_supported()}")

## 2 · Install dependencies
Unsloth + TRL for GRPO, yfinance for real-crash eval.

In [ ]:
%%capture
!pip install --upgrade pip
!pip install unsloth vllm
!pip install --upgrade --no-cache-dir --no-deps git+https://github.com/huggingface/trl.git
!pip install yfinance

## 3 · Clone the repo
Brings `config.py`, `sim.py`, `scenarios.py`, `prompt.py`, `reward.py`, `baselines.py`, `eval.py` into the Colab FS.

In [ ]:
import os, sys, subprocess

REPO_DIR = "/content/qwen-rl-shanghai"
if not os.path.isdir(REPO_DIR):
    subprocess.run([
        "git", "clone",
        "https://github.com/Grinta-Protocol/qwen-rl-shanghai.git",
        REPO_DIR,
    ], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)

sys.path.insert(0, f"{REPO_DIR}/pid_rl")
os.chdir(f"{REPO_DIR}/pid_rl")
!ls

## 4 · Imports — Unsloth + our modules

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
import copy
import numpy as np
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

from config import TRAIN, SIM, GUARD, PID
from sim import Simulator
from scenarios import generate_batch
from prompt import build_prompt, parse_output, ParseError
from reward import compute_reward
from baselines import StaticGainsPolicy, HeuristicPolicy, RandomPolicy

print(f"Base model: {TRAIN.base_model}")
print(f"LoRA rank:  {TRAIN.lora_rank}")
print(f"Max steps:  {TRAIN.max_steps}")
print(f"KL coef:    {TRAIN.kl_coef}")
print(f"bf16:       {is_bfloat16_supported()}")

## 5 · Load Qwen 2.5 1.5B Instruct with LoRA
1.5B in bf16 ≈ 3 GB — fits T4 with room for KV cache + vLLM.

In [ ]:
MAX_SEQ_LEN = TRAIN.max_prompt_length + TRAIN.max_completion_length

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=TRAIN.base_model,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=False,
    fast_inference=True,
    max_lora_rank=TRAIN.lora_rank,
    gpu_memory_utilization=0.6,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=TRAIN.lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=TRAIN.lora_rank,
    use_gradient_checkpointing="unsloth",
    random_state=TRAIN.seed,
)
print("Model + LoRA ready.")

## 6 · Build the training dataset
Pre-generate N scenarios, pre-run each sim to a non-trivial state, snapshot (prompt, simulator). At train time GRPO generates G completions per prompt; the reward fn deep-copies the matching simulator for each completion.

In [ ]:
N_TRAIN_SCENARIOS = 256                  # plenty of diversity for 500 steps
PRE_ROLL_MIN, PRE_ROLL_MAX = 10, 80       # vary the starting state per scenario

scenarios_pool = generate_batch(n=N_TRAIN_SCENARIOS, seed=TRAIN.seed)

sim_pool = []
prompt_messages_pool = []
rng = np.random.default_rng(seed=TRAIN.seed + 1)

for idx, scenario in enumerate(scenarios_pool):
    sim = Simulator(
        btc_path=scenario.btc_path,
        initial_kp=scenario.initial_kp,
        initial_ki=scenario.initial_ki,
        rng=np.random.default_rng(seed=TRAIN.seed + idx),
    )
    pre_roll = int(rng.integers(PRE_ROLL_MIN, PRE_ROLL_MAX))
    history = sim.run_forward(n_steps=pre_roll)
    state = sim.get_state()
    dev_hist = [h.deviation_pct for h in history[-8:]]
    sys_p, user_p = build_prompt(
        state=state,
        btc_history=scenario.btc_path[: sim.step_idx + 1],
        deviation_history=dev_hist,
    )
    sim_pool.append(sim)
    prompt_messages_pool.append([
        {"role": "system", "content": sys_p},
        {"role": "user",   "content": user_p},
    ])

dataset = Dataset.from_list([
    {"prompt": prompt_messages_pool[i], "scenario_idx": i}
    for i in range(len(scenarios_pool))
])
print(dataset)
print("--- first prompt (truncated) ---")
print(prompt_messages_pool[0][1]["content"][:500])

## 7 · Reward function + parse-success diagnostic
We register two reward functions:
1. **`pid_reward`** — the main GRPO signal (deviation / rate_std / action_mag / bounds / monotonic / emergency)
2. **`json_validity`** — small bonus for well-formed JSON (tighter than parse penalty)

TRL averages multiple reward functions, so keeping them separate also gives nicer logging.

In [ ]:
def _completion_text(c):
    if isinstance(c, str):
        return c
    if isinstance(c, list) and c and isinstance(c[0], dict):
        return "\n".join(str(m.get("content", "")) for m in c)
    return str(c)


def pid_reward(prompts, completions, scenario_idx, **kwargs):
    rewards = []
    for completion, idx in zip(completions, scenario_idx):
        text = _completion_text(completion)
        sim = copy.deepcopy(sim_pool[idx])
        br = compute_reward(text, sim)
        rewards.append(float(br.total))
    return rewards


def json_validity(prompts, completions, **kwargs):
    """Small shaping: +0.5 if the output parses, -0.5 otherwise."""
    rewards = []
    for completion in completions:
        text = _completion_text(completion)
        parsed = parse_output(text)
        rewards.append(0.5 if not isinstance(parsed, ParseError) else -0.5)
    return rewards


# Smoke-test both on a static baseline completion
sample = '{"action":"adjust","new_kp":2.5,"new_ki":0.002,"is_emergency":false,"reasoning":"test"}'
print("pid_reward sample:",   pid_reward([None], [sample], [0]))
print("json_validity sample:", json_validity([None], [sample]))

## 8 · GRPO training config

In [ ]:
training_args = GRPOConfig(
    output_dir="outputs",
    use_vllm=True,
    learning_rate=TRAIN.learning_rate,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=TRAIN.warmup_ratio,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    logging_steps=5,
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    per_device_train_batch_size=TRAIN.batch_size,
    gradient_accumulation_steps=1,
    num_generations=TRAIN.num_generations,
    max_prompt_length=TRAIN.max_prompt_length,
    max_completion_length=TRAIN.max_completion_length,
    max_steps=TRAIN.max_steps,
    save_steps=50,
    max_grad_norm=0.1,
    beta=TRAIN.kl_coef,
    seed=TRAIN.seed,
    report_to="none",
)
print(training_args)

## 9 · Create trainer + train
`save_steps=50` → checkpoint every 50 updates. If Colab disconnects you resume from the last one.

In [ ]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[pid_reward, json_validity],
    args=training_args,
    train_dataset=dataset,
)
trainer.train()

## 10 · Save LoRA adapters
Small enough (~20 MB at rank 8) to fit anywhere. Download from the Colab files tab or push to HF Hub.

In [ ]:
model.save_lora("pid_rl_lora_v1")
!ls -la pid_rl_lora_v1

# Optional: push to HF Hub. Uncomment + set token via `from huggingface_hub import login; login()`.
# model.push_to_hub_merged(
#     "Grinta-Protocol/qwen-rl-pid-v1",
#     tokenizer,
#     save_method="lora",
#     token=None,
# )

## 11 · Evaluate vs baselines on synthetic holdout + real crashes

In [ ]:
from eval import run_full_eval


class TrainedModelPolicy:
    """Wraps the trained model to match the baselines.Policy interface."""
    name = "trained"

    def __init__(self, model, tokenizer, max_new_tokens=256, temperature=0.3):
        self.model = model
        self.tokenizer = tokenizer
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        FastLanguageModel.for_inference(self.model)

    def decide(self, state, btc_history, deviation_history):
        sys_p, user_p = build_prompt(
            state=state,
            btc_history=btc_history,
            deviation_history=deviation_history,
        )
        messages = [
            {"role": "system", "content": sys_p},
            {"role": "user",   "content": user_p},
        ]
        text = self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=self.max_new_tokens,
                temperature=self.temperature,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id,
            )
        gen = self.tokenizer.decode(
            out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )
        return gen


trained_policy = TrainedModelPolicy(model, tokenizer)

policies = [
    StaticGainsPolicy(),
    HeuristicPolicy(),
    RandomPolicy(rng=np.random.default_rng(seed=123)),
    trained_policy,
]

results = run_full_eval(
    policies=policies,
    n_synthetic_holdout=24,
    include_real=True,
    verbose=False,
)

## 12 · Quick qualitative peek
Print the model's decision on 3 sample states — crash, pump, stable — to sanity-check it's not drifting monotonically.

In [ ]:
from scenarios import _gen_crash, _gen_pump, _gen_stable

peek_rng = np.random.default_rng(seed=777)
for name, gen in [("crash", _gen_crash), ("pump", _gen_pump), ("stable", _gen_stable)]:
    sc = gen(peek_rng, SIM.episode_steps)
    sim = Simulator(
        btc_path=sc.btc_path, initial_kp=sc.initial_kp, initial_ki=sc.initial_ki,
        rng=np.random.default_rng(seed=7),
    )
    hist = sim.run_forward(n_steps=50)
    dev_hist = [h.deviation_pct for h in hist[-8:]]
    state = sim.get_state()
    completion = trained_policy.decide(
        state=state,
        btc_history=sc.btc_path[: sim.step_idx + 1],
        deviation_history=dev_hist,
    )
    parsed = parse_output(completion)
    print(f"\n=== {name} ===")
    print(f"  btc: {sc.btc_path[sim.step_idx - 5]:.0f} → {sc.btc_path[sim.step_idx]:.0f}")
    print(f"  dev: {state.deviation_pct:+.3f}%   kp={state.kp:.3f}  ki={state.ki:.5f}")
    print(f"  model raw: {completion[:300]}")
    print(f"  parsed:    {parsed}")